# LumaPhoto — Auto Enhance Neural Network Training

**What this notebook does:**
1. Downloads public photo enhancement datasets (DPED + DIV2K + MIT-Adobe FiveK via Kaggle)
2. Trains an EfficientNet-B0 model to predict 15 enhancement parameters
3. Exports `enhancer_params.onnx`
4. Saves to Google Drive so you can download it

**How to run:**
- Runtime → Change runtime type → **T4 GPU** (free tier)
- Runtime → Run all  *(takes ~2-3 hours on T4)*
- Download `enhancer_params.onnx` from the last cell
- Place it next to `LumaPhoto.exe` — the app picks it up automatically

**Datasets used (all free / open access):**
- **DPED** — iPhone vs Canon DSLR pairs (~24 K patches)
- **DIV2K** — 800 high-res images used as synthetic pairs
- **MIT-Adobe FiveK** — 5 000 expert-retouched pairs (via Kaggle, free account needed)
- **PPR10K** — 11 K images × 3 expert retouches (optional, ~30 GB)


In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q timm lpips onnx onnxruntime

In [ ]:
# ── Cell 3: Mount Google Drive (to save the trained model) ────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/LumaPhoto'
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Model will be saved to: {SAVE_DIR}')

In [ ]:
# ── Cell 4: Download DPED (iPhone + Canon) — no login required ────────────────
import urllib.request, zipfile, os

DPED_DIR = '/content/data/dped'
os.makedirs(DPED_DIR, exist_ok=True)

DPED_URLS = {
    'iphone_train': 'https://people.ee.ethz.ch/~ihnatova/dped/iphone_train.zip',
    'canon_train':  'https://people.ee.ethz.ch/~ihnatova/dped/canon_train.zip',
}

for name, url in DPED_URLS.items():
    dest = f'{DPED_DIR}/{name}.zip'
    if not os.path.exists(dest):
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(url, dest)
    print(f'Extracting {name}...')
    with zipfile.ZipFile(dest) as z:
        z.extractall(DPED_DIR)

print('DPED ready.')

In [ ]:
# ── Cell 5: Download DIV2K (high-res photos for synthetic pairs) ──────────────
DIV2K_DIR = '/content/data/div2k'
os.makedirs(DIV2K_DIR, exist_ok=True)

DIV2K_URLS = [
    'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip',
    'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip',
]

for url in DIV2K_URLS:
    name = url.split('/')[-1]
    dest = f'{DIV2K_DIR}/{name}'
    if not os.path.exists(dest):
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(url, dest)
    print(f'Extracting {name}...')
    with zipfile.ZipFile(dest) as z:
        z.extractall(DIV2K_DIR)

print('DIV2K ready.')

In [ ]:
# ── Cell 6: MIT-Adobe FiveK via Kaggle (OPTIONAL but recommended) ─────────────
# Requires a free Kaggle account and API key.
# Skip this cell if you don't have Kaggle — the model trains fine on DPED + DIV2K.
#
# To set up:
#   1. Go to https://www.kaggle.com/settings → API → Create New Token
#   2. Upload the downloaded kaggle.json file when prompted below

USE_FIVEK = True   # set to False to skip

if USE_FIVEK:
    from google.colab import files
    print('Upload your kaggle.json API token:')
    uploaded = files.upload()

    os.makedirs('/root/.config/kaggle', exist_ok=True)
    with open('/root/.config/kaggle/kaggle.json', 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

    !pip install -q kaggle
    !kaggle datasets download -d jerryguo02/mit-adobe-fivek-dataset -p /content/data/fivek --unzip
    FIVEK_ROOT = '/content/data/fivek'
    print('FiveK ready.')
else:
    FIVEK_ROOT = None
    print('Skipping FiveK.')

In [ ]:
# ── Cell 7: Write training source files ───────────────────────────────────────
# Writes pipeline.py, model.py, dataset.py, losses.py inline so the notebook
# is completely self-contained.

# ── pipeline.py ───────────────────────────────────────────────────────────────
PIPELINE_PY = '''
import torch, torch.nn.functional as F

PARAM_NAMES = ["exposure","brilliance","highlights","shadows","contrast",
               "brightness","black_point","saturation","vibrance","warmth",
               "tint","sharpness","definition","noise","vignette"]
NUM_PARAMS = 15
PARAM_RANGES = [(-100,100)]*11 + [(0,100)]*4

def apply_params(img, params):
    B,_,H,W = img.shape
    (exposure,brilliance,highlights,shadows,contrast,brightness,black_point,
     saturation,vibrance,warmth,tint,_sh,_df,_nr,vignette) = params.unbind(1)
    x = img * 255.0
    x = x * torch.pow(2.0, exposure/100.0).view(B,1,1,1)
    x = x + (brightness*1.4).view(B,1,1,1)
    co = (1.0 + contrast/100.0).view(B,1,1,1)
    x = (x - 128.0)*co + 128.0
    lum = x[:,0:1]*0.299 + x[:,1:2]*0.587 + x[:,2:3]*0.114
    hi = torch.clamp((lum-128.0)/127.0, 0, 1)
    sh = torch.clamp((128.0-lum)/128.0, 0, 1)
    x = x + (highlights.view(B,1,1,1)*-0.9)*hi + (shadows.view(B,1,1,1)*1.1)*sh
    lum = x[:,0:1]*0.299 + x[:,1:2]*0.587 + x[:,2:3]*0.114
    x = x + (128.0-lum)*(brilliance.view(B,1,1,1)/100.0)*0.36
    bp = black_point.view(B,1,1,1)
    pos_bp = F.softplus(bp) - F.softplus(torch.zeros_like(bp))
    bl = pos_bp*1.25
    x = (x - bl) * (255.0 / torch.clamp(255.0-bl, min=1.0))
    lum = x[:,0:1]*0.299 + x[:,1:2]*0.587 + x[:,2:3]*0.114
    x = lum + (x-lum)*(1.0 + saturation.view(B,1,1,1)/100.0)
    vib = vibrance.view(B,1,1,1)/100.0
    maxc = x.max(dim=1,keepdim=True)[0]
    avg = x.mean(dim=1,keepdim=True)
    x = avg + (x-avg)*(1.0 + vib*(1.0 - torch.abs(maxc-avg)/128.0))
    w = (warmth*0.55).view(B,1,1,1)
    t = (tint*0.28).view(B,1,1,1)
    x = torch.cat([x[:,0:1]+w+t, x[:,1:2]-t*(0.18/0.28), x[:,2:3]-w+t], dim=1)
    vy = torch.linspace(-1,1,H,device=img.device,dtype=img.dtype)
    vx = torch.linspace(-1,1,W,device=img.device,dtype=img.dtype)
    yy,xx = torch.meshgrid(vy,vx,indexing="ij")
    dist = (xx*xx+yy*yy).sqrt().unsqueeze(0).unsqueeze(0)
    x = x * (1.0 - torch.clamp(dist-0.35,min=0.0)*(vignette.view(B,1,1,1)/85.0))
    return torch.clamp(x/255.0, 0, 1)
'''

# ── model.py ──────────────────────────────────────────────────────────────────
MODEL_PY = '''
import torch, torch.nn as nn, timm
from pipeline import NUM_PARAMS, PARAM_RANGES

class PhotoEnhancerNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model("efficientnet_b0", pretrained=pretrained,
                                           num_classes=0, global_pool="avg")
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(0.35), nn.Linear(feat_dim, 512), nn.SiLU(),
            nn.LayerNorm(512), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.SiLU(), nn.Linear(256, NUM_PARAMS))
        self._sym = [i for i,(lo,_) in enumerate(PARAM_RANGES) if lo<0]
        self._pos = [i for i,(lo,_) in enumerate(PARAM_RANGES) if lo==0]
    def forward(self, x):
        raw = self.head(self.backbone(x))
        out = torch.empty_like(raw)
        out[:,self._sym] = torch.tanh(raw[:,self._sym])*100.0
        out[:,self._pos] = torch.sigmoid(raw[:,self._pos])*100.0
        return out
'''

with open('/content/pipeline.py','w') as f: f.write(PIPELINE_PY)
with open('/content/model.py',   'w') as f: f.write(MODEL_PY)
print('Source files written.')

In [ ]:
# ── Cell 8: Dataset classes ────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content')

import random
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, ConcatDataset
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _to_tensor(img):
    return transforms.functional.to_tensor(img)

def _resize_min(img, min_side):
    w, h = img.size
    if min(w,h) < min_side:
        s = min_side / min(w,h)
        img = img.resize((int(w*s), int(h*s)), Image.BICUBIC)
    return img

def _paired_aug(inp, tgt, crop=256):
    seed = random.randint(0, 2**31)
    aug = transforms.Compose([transforms.RandomCrop(crop), transforms.RandomHorizontalFlip()])
    torch.manual_seed(seed); random.seed(seed)
    i = aug(_to_tensor(inp))
    torch.manual_seed(seed); random.seed(seed)
    t = aug(_to_tensor(tgt))
    return i, t

class DPEDDataset(Dataset):
    def __init__(self, root, phone='iphone', dslr='canon', crop=128):
        pd = Path(root)/phone/'train'
        dd = Path(root)/dslr/'train'
        pf = {f.name for f in pd.iterdir() if f.suffix=='.jpg'}
        df = {f.name for f in dd.iterdir() if f.suffix=='.jpg'}
        self.pairs = [(pd/n, dd/n) for n in sorted(pf & df)]
        self.crop = crop
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        pp, dp = self.pairs[i]
        inp = _to_tensor(Image.open(pp).convert('RGB'))
        tgt = _to_tensor(Image.open(dp).convert('RGB'))
        if inp.shape[-1] >= self.crop and inp.shape[-2] >= self.crop:
            r = random.randint(0, inp.shape[-2]-self.crop)
            c = random.randint(0, inp.shape[-1]-self.crop)
            inp = inp[:, r:r+self.crop, c:c+self.crop]
            tgt = tgt[:, r:r+self.crop, c:c+self.crop]
            if random.random() > 0.5: inp,tgt = torch.flip(inp,[-1]),torch.flip(tgt,[-1])
        return inp, tgt

class FiveKDataset(Dataset):
    def __init__(self, root, expert='expertC', crop=256):
        self.inp_dir = Path(root)/'input'
        self.tgt_dir = Path(root)/expert
        self.files = sorted(f.name for f in self.inp_dir.iterdir()
                            if f.suffix.lower() in {'.jpg','.jpeg','.png'})
        self.crop = crop
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        n = self.files[i]
        inp = Image.open(self.inp_dir/n).convert('RGB')
        tgt = Image.open(self.tgt_dir/n).convert('RGB')
        inp = _resize_min(inp, self.crop+32)
        tgt = tgt.resize(inp.size, Image.BICUBIC)
        return _paired_aug(inp, tgt, self.crop)

class SyntheticDataset(Dataset):
    EXTS = {'.jpg','.jpeg','.png'}
    def __init__(self, image_dir, crop=256):
        self.files = sorted(p for p in Path(image_dir).rglob('*') if p.suffix.lower() in self.EXTS)
        self.aug = transforms.Compose([transforms.RandomCrop(crop), transforms.RandomHorizontalFlip()])
    def __len__(self): return len(self.files)
    def _degrade(self, x):
        x = x * (2.0 ** random.uniform(-1.5, 1.5))
        co = random.uniform(0.6, 1.6)
        x = (x - 0.5)*co + 0.5
        lum = x[0:1]*0.299 + x[1:2]*0.587 + x[2:3]*0.114
        x = lum + (x-lum)*random.uniform(0.3, 1.8)
        wt = random.uniform(-0.15, 0.15)
        x[0] += wt; x[2] -= wt
        if random.random() < 0.3: x = x - random.uniform(0, 0.12)
        if random.random() < 0.3: x = x * random.uniform(1.1, 1.4)
        return torch.clamp(x, 0, 1)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB')
        img = _resize_min(img, 288)
        seed = random.randint(0, 2**31)
        torch.manual_seed(seed); random.seed(seed)
        clean = self.aug(_to_tensor(img))
        return self._degrade(clean.clone()), clean

# Build combined dataset
datasets = []
try:
    dped_ds = DPEDDataset('/content/data/dped')
    datasets.append(dped_ds)
    print(f'DPED: {len(dped_ds):,} pairs')
except Exception as e:
    print(f'DPED skipped: {e}')

try:
    div2k_ds = SyntheticDataset('/content/data/div2k')
    datasets.append(div2k_ds)
    print(f'DIV2K synthetic: {len(div2k_ds):,} images')
except Exception as e:
    print(f'DIV2K skipped: {e}')

if FIVEK_ROOT:
    try:
        fivek_ds = FiveKDataset(FIVEK_ROOT)
        datasets.append(fivek_ds)
        print(f'FiveK: {len(fivek_ds):,} pairs')
    except Exception as e:
        print(f'FiveK skipped: {e}')

assert datasets, 'No datasets loaded!'
train_ds = ConcatDataset(datasets) if len(datasets) > 1 else datasets[0]
print(f'\nTotal training pairs: {len(train_ds):,}')

In [ ]:
# ── Cell 9: Build model + loss ────────────────────────────────────────────────
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tv_models
from torch.utils.data import DataLoader
from torchvision import transforms
from pipeline import apply_params
from model import PhotoEnhancerNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')

IMAGENET_MEAN = torch.tensor([0.485,0.456,0.406],device=DEVICE).view(1,3,1,1)
IMAGENET_STD  = torch.tensor([0.229,0.224,0.225],device=DEVICE).view(1,3,1,1)

def to_backbone_input(img):
    small = F.interpolate(img, size=(224,224), mode='bilinear', align_corners=False)
    return (small - IMAGENET_MEAN) / IMAGENET_STD

# ── VGG perceptual loss ───────────────────────────────────────────────────────
class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = tv_models.vgg19(weights=tv_models.VGG19_Weights.IMAGENET1K_V1).features
        self.s1 = vgg[:9].eval()
        self.s2 = vgg[:18].eval()
        for p in self.parameters(): p.requires_grad_(False)
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def _n(self,x): return (x-self.mean)/self.std
    def forward(self, p, t):
        p,t = self._n(p), self._n(t)
        return F.l1_loss(self.s1(p),self.s1(t)) + F.l1_loss(self.s2(p),self.s2(t))*0.5

model   = PhotoEnhancerNet(pretrained=True).to(DEVICE)
vgg_loss = VGGLoss().to(DEVICE)

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 3e-5},
    {'params': model.head.parameters(),     'lr': 3e-4},
], weight_decay=1e-4)

BATCH  = 24
EPOCHS = 40
loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
sched  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=max(1,EPOCHS//3))
scaler = torch.cuda.amp.GradScaler()
print(f'Batches per epoch: {len(loader):,}  |  ETA ~{len(loader)*EPOCHS*0.2/3600:.1f} h on T4')

In [ ]:
# ── Cell 10: Training loop ────────────────────────────────────────────────────
import math, time

CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

best_loss = math.inf

for epoch in range(EPOCHS):
    model.train()
    ep_loss = 0.0
    t0 = time.time()

    for step, (inp, tgt) in enumerate(loader):
        inp = inp.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            params = model(to_backbone_input(inp))
            pred   = apply_params(inp, params)
            l1   = F.l1_loss(pred, tgt)
            perc = vgg_loss(pred, tgt)
            reg  = (params.abs()/100.0).mean() * 0.01
            loss = l1 + 0.1*perc + reg

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        ep_loss += loss.item()

    sched.step()
    avg = ep_loss / len(loader)
    elapsed = time.time() - t0
    print(f'Epoch {epoch+1:03d}/{EPOCHS}  loss={avg:.4f}  ({elapsed:.0f}s)')

    torch.save(model.state_dict(), f'{CKPT_DIR}/last.pt')
    if avg < best_loss:
        best_loss = avg
        torch.save(model.state_dict(), f'{CKPT_DIR}/best.pt')
        print(f'  ↳ New best saved')

print(f'\nTraining complete. Best loss: {best_loss:.4f}')

In [ ]:
# ── Cell 11: Export to ONNX ───────────────────────────────────────────────────
import torch.onnx

model.eval()
model.load_state_dict(torch.load(f'{CKPT_DIR}/best.pt', map_location='cpu'))
model = model.cpu()

dummy     = torch.randn(1, 3, 224, 224)
onnx_path = '/content/enhancer_params.onnx'

with torch.no_grad():
    torch.onnx.export(
        model, dummy, onnx_path,
        opset_version=17,
        input_names=['input'],
        output_names=['params'],
        dynamic_axes={'input':{0:'batch'}, 'params':{0:'batch'}},
        do_constant_folding=True,
    )

print(f'Exported: {onnx_path}')

# Verify
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession(onnx_path)
out  = sess.run(None, {'input': np.random.randn(1,3,224,224).astype(np.float32)})[0]
print(f'Output shape: {out.shape}')
from pipeline import PARAM_NAMES
print('Sample output:')
for name, val in zip(PARAM_NAMES, out[0]):
    print(f'  {name:12s}: {val:+.1f}')
print('\nONNX export verified ✓')

In [ ]:
# ── Cell 12: Save to Google Drive + download ──────────────────────────────────
import shutil
drive_path = f'{SAVE_DIR}/enhancer_params.onnx'
shutil.copy(onnx_path, drive_path)
print(f'Saved to Google Drive: {drive_path}')

# Also offer direct download
from google.colab import files
print('\nDownloading enhancer_params.onnx ...')
files.download(onnx_path)
print('\nDone! Place enhancer_params.onnx next to LumaPhoto.exe')
print('The app will detect it automatically on next launch.')